In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
load_dotenv()


True

In [4]:
llm = ChatDeepSeek(
    model="deepseek-chat"
)

In [5]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [6]:
def generate_joke(state: JokeState):
    prompt = f'generate a joke on the topic {state['topic']}'
    response = llm.invoke(prompt).content
    return {'joke': response}

In [7]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [8]:
graph = StateGraph(JokeState)

In [9]:
graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

In [10]:
checkpointer = InMemorySaver()

In [11]:
agent = graph.compile(checkpointer = checkpointer)

In [21]:
config_1 = { "configurable": {"thread_id": "1"}}

In [13]:
agent.invoke({'topic': 'pizza'}, config = config_1)

{'topic': 'pizza',
 'joke': "Why don't pizzas ever get into arguments?\n\nBecause they always know how to *slice* the issue... and they're too cheesy to stay mad.",
 'explanation': 'Here’s the breakdown of why this joke works:\n\n**1. The Setup (a classic pun structure)**  \nThe question sets up an expectation that the answer will be a logical reason (e.g., "because they have no backbone"). Instead, it puns on pizza-related words.\n\n**2. The First Punchline: “Slice the issue”**  \n- **Literal meaning:** A pizza is cut into slices.  \n- **Figurative meaning:** To “slice” an issue means to break it down into manageable parts or analyze it clearly.  \nSo the joke says pizzas are great at resolving conflicts because they’d *literally* slice the argument apart—like cutting a pizza. It’s a clever visual pun that flips a common idiom.\n\n**3. The Second Punchline: “Too cheesy to stay mad”**  \n- **Literal meaning:** Pizza is covered in cheese.  \n- **Figurative meaning:** “Cheesy” means over

In [14]:
agent.get_state(config_1)

StateSnapshot(values={'topic': 'pizza', 'joke': "Why don't pizzas ever get into arguments?\n\nBecause they always know how to *slice* the issue... and they're too cheesy to stay mad.", 'explanation': 'Here’s the breakdown of why this joke works:\n\n**1. The Setup (a classic pun structure)**  \nThe question sets up an expectation that the answer will be a logical reason (e.g., "because they have no backbone"). Instead, it puns on pizza-related words.\n\n**2. The First Punchline: “Slice the issue”**  \n- **Literal meaning:** A pizza is cut into slices.  \n- **Figurative meaning:** To “slice” an issue means to break it down into manageable parts or analyze it clearly.  \nSo the joke says pizzas are great at resolving conflicts because they’d *literally* slice the argument apart—like cutting a pizza. It’s a clever visual pun that flips a common idiom.\n\n**3. The Second Punchline: “Too cheesy to stay mad”**  \n- **Literal meaning:** Pizza is covered in cheese.  \n- **Figurative meaning:** 

In [15]:
list(agent.get_state_history(config_1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Why don't pizzas ever get into arguments?\n\nBecause they always know how to *slice* the issue... and they're too cheesy to stay mad.", 'explanation': 'Here’s the breakdown of why this joke works:\n\n**1. The Setup (a classic pun structure)**  \nThe question sets up an expectation that the answer will be a logical reason (e.g., "because they have no backbone"). Instead, it puns on pizza-related words.\n\n**2. The First Punchline: “Slice the issue”**  \n- **Literal meaning:** A pizza is cut into slices.  \n- **Figurative meaning:** To “slice” an issue means to break it down into manageable parts or analyze it clearly.  \nSo the joke says pizzas are great at resolving conflicts because they’d *literally* slice the argument apart—like cutting a pizza. It’s a clever visual pun that flips a common idiom.\n\n**3. The Second Punchline: “Too cheesy to stay mad”**  \n- **Literal meaning:** Pizza is covered in cheese.  \n- **Figurative meaning:**

### Time Travel

In [18]:
# 1f1a110a-d89b-6a7e-8000-f6d604d9b811
agent.get_state({"configurable": {"thread_id":"1","checkpoint_id":"1f1a110a-d89b-6a7e-8000-f6d604d9b811"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1a110a-d89b-6a7e-8000-f6d604d9b811'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-08-26T05:40:42.822924+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1a110a-d899-637f-bfff-d281fc223d48'}}, tasks=(PregelTask(id='246fc630-cec5-402b-d386-d7e6301dcecd', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': "Why don't pizzas ever get into arguments?\n\nBecause they always know how to *slice* the issue... and they're too cheesy to stay mad."}),), interrupts=())

In [19]:
agent.invoke(None,{"configurable":{"thread_id":"1","checkpoint_id": "1f1a110a-d89b-6a7e-8000-f6d604d9b811"}})

{'topic': 'pizza',
 'joke': 'Here’s a joke for you:\n\n**Why did the pizza go to the doctor?**\n\n**Because it was feeling a little “crusty” and couldn’t find its “slice” of peace!** 🍕\n\nHope that made you *dough* with laughter!',
 'explanation': 'This joke works on three levels of wordplay, all tied to pizza vocabulary:\n\n1. **“Crusty”** – This is a double meaning. Literally, a pizza has a crust (the outer edge). Figuratively, “crusty” means irritable, grumpy, or unwell (like when someone is in a bad mood). So the pizza is feeling sick and annoyed.\n\n2. **“Couldn’t find its ‘slice’ of peace”** – This twists the common phrase **“a piece of peace”** (meaning a bit of calm or happiness). By changing **“piece”** to **“slice”** (a pizza slice), the joke keeps the pizza theme while suggesting the pizza is anxious or restless and just wants some calm.\n\n3. **The punchline tag: “Hope that made you *dough* with laughter!”** – This puns on **“double”** (as in *double over* with laughter) by

In [23]:
list(agent.get_state_history(config_1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Here’s a joke for you:\n\n**Why did the pizza go to the doctor?**\n\n**Because it was feeling a little “crusty” and couldn’t find its “slice” of peace!** 🍕\n\nHope that made you *dough* with laughter!', 'explanation': 'This joke works on three levels of wordplay, all tied to pizza vocabulary:\n\n1. **“Crusty”** – This is a double meaning. Literally, a pizza has a crust (the outer edge). Figuratively, “crusty” means irritable, grumpy, or unwell (like when someone is in a bad mood). So the pizza is feeling sick and annoyed.\n\n2. **“Couldn’t find its ‘slice’ of peace”** – This twists the common phrase **“a piece of peace”** (meaning a bit of calm or happiness). By changing **“piece”** to **“slice”** (a pizza slice), the joke keeps the pizza theme while suggesting the pizza is anxious or restless and just wants some calm.\n\n3. **The punchline tag: “Hope that made you *dough* with laughter!”** – This puns on **“double”** (as in *double ove

In [25]:
# 1f1a110a-d89b-6a7e-8000-f6d604d9b811
agent.update_state({"configurable":{"thread_id":"1","checkpoint_id": "1f1a110a-d89b-6a7e-8000-f6d604d9b811", "checkpoint_ns": ""}},{"topic": "samosa"})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1a1137-6899-624b-8001-fb825e118806'}}

In [26]:
agent.invoke(None,{"configurable":{"thread_id":"1","checkpoint_id": "1f1a1137-6899-624b-8001-fb825e118806"}})

{'topic': 'samosa',
 'joke': 'Here’s a joke for you:\n\n**Why did the samosa break up with the chutney?**  \nBecause the chutney was too clingy, and the samosa felt it was being *dipped* into too much drama!  \n\n---\n\nAnd here’s a bonus one-liner:  \n**I told my samosa to be more open-minded. Now it’s a spring roll.**  \n\nHope that brought a crispy smile! 😄',
 'explanation': 'Here’s the breakdown of both jokes:\n\n---\n\n**Joke 1: “Why did the samosa break up with the chutney?”**  \n**Punchline:** *“Because the chutney was too clingy, and the samosa felt it was being dipped into too much drama!”*\n\n**Explanation:**  \n- **Literal vs. figurative:** Samosas are *literally* dipped into chutney when eaten. The joke plays on the word **“dipped”**—meaning both the physical action (dipping the samosa into chutney) and the figurative idea of being “involved” or “submerged” in something (like drama).  \n- **“Clingy”** is a pun: Chutney is thick and sticks to the samosa (clingy texture), but

### fault tolerance

In [16]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [17]:
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [18]:
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

In [19]:
def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

In [20]:
def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [21]:
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

In [22]:
checkpointer = InMemorySaver()

In [23]:
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


In [ ]:
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})